# 4. Baseline Model

For our baseline model, and given that we are working on a regression problem, we will fit an unregularized linear regression to our preprocessed data.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [8]:
df = pd.read_csv('../data/preprocessed.csv')

In [11]:
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

X = df.drop(columns=['log_shares'])
y = df['log_shares']

kf = KFold(n_splits=10, shuffle=True, random_state=42)

mse_scores = []
mae_scores = []
r2_scores = []

for train_index, test_index in kf.split(X):
    scaler = StandardScaler()
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # scale
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # predictions in log space
    y_pred_log = model.predict(X_test)

    # invert transform safely
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)

    mse_scores.append(mean_squared_error(y_true, y_pred))
    mae_scores.append(mean_absolute_error(y_true, y_pred))
    r2_scores.append(r2_score(y_true, y_pred))

print(f'Baseline MSE: {np.mean(mse_scores)}')
print(f'Baseline MAE: {np.mean(mae_scores)}')
print(f'Baseline R2: {np.mean(r2_scores)}')

Baseline MSE: 128125425.62975514
Baseline MAE: 2302.2307893362954
Baseline R2: -0.008420254289900564


In [13]:
# Fit a Random Forest Regressor to the data
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on the test set
y_pred_rf_log = rf_model.predict(X_test)
y_pred_rf = np.expm1(y_pred_rf_log)
y_true_rf = np.expm1(y_test)

# Evaluate the Random Forest model
rf_mse = mean_squared_error(y_true_rf, y_pred_rf)
rf_mae = mean_absolute_error(y_true_rf, y_pred_rf)
rf_r2 = r2_score(y_true_rf, y_pred_rf)

print(f'Random Forest MSE: {rf_mse}')
print(f'Random Forest MAE: {rf_mae}')
print(f'Random Forest R2: {rf_r2}')

Random Forest MSE: 74328959.65938245
Random Forest MAE: 2206.257815854084
Random Forest R2: -0.0018694004666979236
